# Train the deepfake detector on Colab

Run with a GPU: **Runtime → Change runtime type → T4 GPU**.

**Before starting**, upload to Google Drive, in a folder called `deepfake` (`MyDrive/deepfake/`):
- `faces.zip`: Celeb-DF face crops (`data/faces.zip` in the repo folder)
- `faces_ffpp.zip`: FaceForensics++ face crops (`data/faces_ffpp.zip`), if you've built them

Everything this notebook produces (checkpoints, metrics, plots) is saved back to `MyDrive/deepfake/`.

**Three experiments.** Each one takes about an hour, so run them in separate sittings if Colab disconnects. Results are saved to Drive after each one, so you never lose finished work.

| | Train on | Test on | What it shows |
|---|---|---|---|
| A | Celeb-DF | Celeb-DF | Baseline: performance on the same kind of data it learned from |
| B | FF++ | FF++ **and** Celeb-DF | **Generalization**: how much performance drops on an unseen dataset |
| C | FF++ + Celeb-DF | both | The model you deploy |

## Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/deepfake"
REPO_URL = "https://github.com/YOUR_USERNAME/deepfake-detector.git"  # <- change this

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.exists("/content/deepfake-detector"):
    !git clone -q $REPO_URL /content/deepfake-detector
%cd /content/deepfake-detector
!git pull -q
!pip install -q -r requirements-train.txt

Copy the face crops from Drive to the Colab disk. Reading tens of thousands of small files straight from Drive is very slow.

In [ ]:
for name in ["faces", "faces_ffpp"]:
    zip_path = f"{DRIVE_DIR}/{name}.zip"
    if os.path.exists(zip_path) and not os.path.exists(f"/content/{name}"):
        !cp "$zip_path" /content/ && unzip -q -o /content/{name}.zip -d /content/ && rm /content/{name}.zip
    print(name, "ready" if os.path.exists(f"/content/{name}/manifest.csv") else "MISSING")

## Experiment A: Celeb-DF only (baseline)

Each epoch prints train loss, validation loss, and validation AUC (frame and video level). The best epoch is kept, and a calibration temperature is fit at the end.

In [ ]:
!python -m model.train --data-dir /content/faces --arch resnet50 --epochs 6 --batch-size 64 \
    --checkpoint-path "$DRIVE_DIR/checkpoints/A_celebdf.pth" --reports-dir "$DRIVE_DIR/reports/A_celebdf"
!python -m model.eval --data-dir /content/faces --checkpoint-path "$DRIVE_DIR/checkpoints/A_celebdf.pth" \
    --reports-dir "$DRIVE_DIR/reports/A_celebdf"

## Experiment B: train on FF++, test on FF++ and on unseen Celeb-DF

This is the key result. Expect a clear drop on Celeb-DF. How big the drop is, and which FF++ methods are hardest, are the things worth writing about.

In [ ]:
!python -m model.train --data-dir /content/faces_ffpp --arch resnet50 --epochs 6 --batch-size 64 \
    --checkpoint-path "$DRIVE_DIR/checkpoints/B_ffpp.pth" --reports-dir "$DRIVE_DIR/reports/B_ffpp"

In [ ]:
# In-dataset: FF++ test split, with a per-method breakdown
!python -m model.eval --data-dir /content/faces_ffpp --checkpoint-path "$DRIVE_DIR/checkpoints/B_ffpp.pth" \
    --reports-dir "$DRIVE_DIR/reports/B_ffpp/test_ffpp"
# Cross-dataset: Celeb-DF test split, which the model has never seen
!python -m model.eval --data-dir /content/faces --checkpoint-path "$DRIVE_DIR/checkpoints/B_ffpp.pth" \
    --reports-dir "$DRIVE_DIR/reports/B_ffpp/test_celebdf"

## Experiment C: train on both (the model you deploy)

Evaluated on both test sets together. The breakdown table shows each dataset separately.

In [ ]:
!python -m model.train --data-dir /content/faces /content/faces_ffpp --arch resnet50 --epochs 6 --batch-size 64 \
    --checkpoint-path "$DRIVE_DIR/checkpoints/C_combined.pth" --reports-dir "$DRIVE_DIR/reports/C_combined"
!python -m model.eval --data-dir /content/faces /content/faces_ffpp --checkpoint-path "$DRIVE_DIR/checkpoints/C_combined.pth" \
    --reports-dir "$DRIVE_DIR/reports/C_combined"

## Look at the plots

In [ ]:
from IPython.display import Image, display
for exp in ["A_celebdf", "B_ffpp/test_celebdf", "C_combined"]:
    for name in ["roc_curve.png", "reliability_diagram.png"]:
        path = f"{DRIVE_DIR}/reports/{exp}/{name}"
        if os.path.exists(path):
            print(exp, name); display(Image(path, width=420))

## Next

1. Download `MyDrive/deepfake/checkpoints/C_combined.pth`, rename it to `best.pth`, and put it in `model/checkpoints/` in the repo. That's the model the API serves.
2. Download the whole `MyDrive/deepfake/reports/` folder into the repo's `reports/` so the README can show the plots.
3. Copy the printed tables from experiments A, B and C into the README results section.